# Stage 2 - AG-MTCNet Multi-Task Fine-Tuning (Best-Checkpoint Kaggle Pipeline)

## Anatomically-Gated Multi-Task Consensus Network with Validation-Selected Checkpoint

---

This notebook produces the final `ag_mtcnet_best.pth` checkpoint for the LungRay platform.

### What this version optimizes
1. **Decoupled anatomical prior**: a frozen Stage 1 Dense-UNet gate supplies stable lung bounds without gradient interference.
2. **Shared encoder tri-head**: DenseNet-121 with parameter-free LayerNorm + Dropout(0.20) for Head A (14 NIH pathologies), Head B (TB screening) and Head C (COVIDx CXR-4 3-class triage).
3. **Masked multi-task loss**: Head A = Asymmetric Focal Loss, Head B = weighted BCEWithLogits, Head C = class-weighted CrossEntropy. Each sample updates only its own head.
4. **Balanced epoch**: fixed 60% NIH : 25% COVID : 15% TB batch composition.
5. **Best-checkpoint selection by validation AUC**, not training loss, with patience-based early stopping.
6. **EMA + raw weights are both evaluated**; the better state is exported to disk.
7. **Layer-wise differential learning rates** for the shared encoder plus head learning rates.
8. **Class-balanced losses**: Head C inverse-frequency CE weights and Head B TB prevalence weighting.
9. **Depth-safe input discovery** for every Kaggle mount layout (`/kaggle/input/<slug>`, `/kaggle/input/<user>/<slug>`, `/kaggle/input/datasets/<user>/<slug>`).
10. **Calibrated decision thresholds** (Youden's J) and held-out test metrics are exported with the checkpoint.

### Required Kaggle Inputs
| Task | Recommended Kaggle dataset slug | Required content |
|:---|:---|:---|
| Stage 1 gate | `stage1-dense-unet` | `dense_unet_best.pth` |
| Warm-start | `best-model` | `best_model.pth` (converged 3-head baseline) |
| NIH ChestX-ray14 | `lungray-nih-512` or `nih-512` | resized images; `dataset/splits/patient_splits.csv` is bundled in the repo |
| COVID | `lungray-covidx-512` (preferred) or `covid19-radiography-database` | `train.txt`/`test.txt`, or class folders |
| Tuberculosis | `tuberculosis-tb-chest-xray-dataset` | `Normal/` and `Tuberculosis/` class folders |

`montgo-shenzhen-resize` is the Stage 1 segmentation export (images + masks); it is not a TB classifier. If only that dataset is mounted, Head B falls back to SYNTHETIC and the notebook prints a diagnostic.

### Outputs
- `checkpoints/ag_mtcnet_best.pth` - deployable payload with `model_state_dict`, validation/test metrics, thresholds and config
- `checkpoints/ag_mtcnet_thresholds.json` - Youden-J thresholds for clinical use
- `checkpoints/ag_mtcnet_training_curves.png` - live training/validation curve
- `/kaggle/working/ag_mtcnet_best.pth` - Kaggle copy for immediate download

### Runtime expectation
At 256 px on T4 x2, `deep` (6,000 samples/epoch) is roughly 15-25 minutes for the full 20-epoch schedule. At 512 px with batch 8 and gradient accumulation, expect 45-70 minutes.

In [ ]:
# ---------------------------------------------------------------------------
# 0. Cross-environment bootstrap, configuration and safe checkpoint helpers
# ---------------------------------------------------------------------------
import os
import sys
import json
import time
import random
import shlex
import shutil
import subprocess
import contextlib
from pathlib import Path

REPO_URL = "https://github.com/philipsgn/LungRay.git"
SEED = 42

IMG_SIZE = int(os.environ.get("IMG_SIZE", "256"))
if IMG_SIZE >= 512:
    BATCH_SIZE = 8
    GRAD_ACCUM_STEPS = 2
else:
    BATCH_SIZE = 20
    GRAD_ACCUM_STEPS = 1

TRAINING_PROFILE = os.environ.get("TRAINING_PROFILE", "deep").strip().lower()
PROFILE_SAMPLES = {"fast": 3000, "deep": 6000, "full": 10000}
SAMPLES_PER_EPOCH = PROFILE_SAMPLES.get(TRAINING_PROFILE, 6000)

PROFILE_POOL_LIMITS = {
    "fast": {"nih": 12000, "covid": 6000, "tb": 4000},
    "deep": {"nih": 40000, "covid": 12000, "tb": 4000},
    "full": {"nih": None, "covid": None, "tb": None},
}
POOL_LIMITS = PROFILE_POOL_LIMITS.get(TRAINING_PROFILE, PROFILE_POOL_LIMITS["deep"])

BATCHES_PER_EPOCH = max(1, SAMPLES_PER_EPOCH // BATCH_SIZE)
EPOCHS_STAGE1 = 2
EPOCHS_STAGE2 = 18
MAX_VAL_BATCHES = 50
PATIENCE = 6
MIN_DELTA = 1e-4
NUM_WORKERS = int(os.environ.get("LUNGRAY_NUM_WORKERS", "2"))
TASK_RATIOS = (0.60, 0.25, 0.15)
IS_KAGGLE = Path("/kaggle/input").exists() or Path("/kaggle/working").exists()
USE_EMA = os.environ.get("LUNGRAY_EMA", "1") == "1"
EMA_DECAY = float(os.environ.get("LUNGRAY_EMA_DECAY", "0.999"))
USE_IMAGENET = os.environ.get("LUNGRAY_IMAGENET", "1") == "1"
REQUIRE_GATE = os.environ.get("LUNGRAY_REQUIRE_GATE", "0") == "1"
RESUME = os.environ.get("LUNGRAY_RESUME", "1") == "1"
NUM_CLASSES_C = 3


def run(cmd, check=True, cwd=None):
    cmd = [str(c) for c in cmd]
    print("$", " ".join(shlex.quote(c) for c in cmd), flush=True)
    proc = subprocess.Popen(
        cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    if check and rc != 0:
        raise RuntimeError(f"Command failed (rc={rc})")
    return rc


if IS_KAGGLE:
    WORK = Path("/kaggle/working")
    WORK.mkdir(parents=True, exist_ok=True)
    REPO = WORK / "Chest-X-rays"
    if not REPO.exists():
        run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)])
    sys.path.insert(0, str(REPO))
    os.chdir(REPO)
else:
    here = Path.cwd().resolve()
    REPO = next(
        (p for p in [here, *here.parents] if (p / "src" / "models" / "multi_task_net.py").exists()),
        here,
    )
    WORK = REPO
    sys.path.insert(0, str(REPO))

print("[OK] Repository root:", REPO)

for module_name, package_name in (
    ("cv2", "opencv-python-headless"),
    ("pandas", "pandas"),
    ("sklearn", "scikit-learn"),
    ("matplotlib", "matplotlib"),
    ("timm", "timm"),
):
    try:
        __import__(module_name)
    except ImportError:
        run([sys.executable, "-m", "pip", "install", "-q", package_name])

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, Sampler

from src.data.dataset import NIH_PATHOLOGY_LIST
from src.data.ingestion import (
    build_recursive_file_map,
    find_covidx_manifests,
    locate_file,
    resolve_dataset_paths,
)
from src.data.make_splits import split_nih_by_patient_id
from src.models.asymmetric_loss import AsymmetricFocalLoss
from src.models.dense_unet import DenseUNet, apply_dilated_soft_mask_tensor
from src.models.losses import MaskedMultiTaskLoss
from src.models.multi_task_net import MultiTaskChestXrayNet
from src.models.trainer import multitask_collate_fn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
if USE_AMP:
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cuda.matmul.allow_tf32 = True
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)


def safe_torch_load(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)
    except Exception:
        return torch.load(path, map_location=map_location, weights_only=False)


def find_input_files(fragment, max_depth=6, limit=20):
    root = Path("/kaggle/input")
    if not root.exists():
        return []
    fragment = fragment.lower()
    pruned = {
        ".git", "__pycache__", "images", "masks", "images_512", "masks_512",
        "data_512", "train", "test", "valid", "val",
    }
    found = []
    base_parts = len(root.parts)
    for dirpath, dirnames, filenames in os.walk(root):
        current = Path(dirpath)
        if len(current.parts) - base_parts >= max_depth:
            dirnames[:] = []
        dirnames[:] = [d for d in dirnames if d.lower() not in pruned]
        for name in filenames:
            low = name.lower()
            if fragment in low and low.endswith((".pth", ".pt", ".ckpt")):
                found.append(str(current / name))
                if len(found) >= limit:
                    return sorted(found)
    return sorted(found)


def amp_autocast():
    if not USE_AMP:
        return contextlib.nullcontext()
    if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):
        return torch.amp.autocast("cuda", enabled=True)
    return torch.cuda.amp.autocast(enabled=True)


def build_grad_scaler():
    if USE_AMP and hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        try:
            return torch.amp.GradScaler("cuda", enabled=True)
        except TypeError:
            pass
    return torch.cuda.amp.GradScaler(enabled=USE_AMP)


print(f"[*] Execution hardware: {device} (CUDA available: {torch.cuda.is_available()})")
print(f"[*] Profile={TRAINING_PROFILE} | {SAMPLES_PER_EPOCH} samples/epoch | {BATCHES_PER_EPOCH} batches | {EPOCHS_STAGE1 + EPOCHS_STAGE2} epochs")
print(f"[*] AMP={USE_AMP} | EMA={USE_EMA} | ImageNet fallback={USE_IMAGENET} | Gate required={REQUIRE_GATE}")

## 1. Dataset resolution and zero-leakage split construction

The resolver scans `/kaggle/input` for the canonical mounts and falls back to the local `dataset/` tree. Every loader prints its provenance so a silently synthetic task is visible before training.

- **NIH ChestX-ray14**: the bundled `dataset/splits/patient_splits.csv` is used first; otherwise a 70/10/20 split is regenerated from `Data_Entry_2017.csv` by `Patient_ID`. Patient isolation is asserted before training (Rule 2).
- **COVIDx CXR-4**: official `train.txt` / `test.txt` manifests are parsed first, preserving the 3-class label and clinical `data_source`. Validation is carved out at patient level and stratified by `(class, source)` (Rule 3). The legacy `covid_splits.csv` is a documented fallback.
- **Tuberculosis**: `tb_splits.csv` is used when the TB image mount is present; otherwise `Normal/` and `Tuberculosis/` class folders are discovered recursively (supports nested `TB_Chest_Radiography_Database/Normal` layouts).
- **Missing tasks** are replaced by a synthetic generator so the notebook never stops with `FileNotFoundError`, but the replacement is printed loudly.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Ingestion helpers: real images when mounted, explicit synthetic fallback
# ---------------------------------------------------------------------------
IMAGE_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def iter_images(root, max_depth=6, limit=30000):
    root = Path(root)
    if not root.exists():
        return
    base = len(root.parts)
    found = 0
    for dirpath, dirnames, filenames in os.walk(root):
        here = Path(dirpath)
        if len(here.parts) - base > max_depth:
            dirnames[:] = []
            continue
        for name in filenames:
            if Path(name).suffix.lower() in IMAGE_EXTS:
                yield here / name
                found += 1
                if found >= limit:
                    return


def resolve_image(image_dir, name, kind, cache):
    if image_dir is None:
        return None
    image_dir = Path(image_dir)
    name = Path(str(name)).name
    stem = Path(name).stem
    variants = [name]
    for ext in IMAGE_EXTS:
        candidate_name = stem + ext
        if candidate_name != name:
            variants.append(candidate_name)

    directories = [image_dir, image_dir / "images"]
    if kind == "nih":
        directories += [image_dir / f"images_{i:03d}" / "images" for i in range(1, 13)]
    elif kind == "covid":
        directories += [
            image_dir / "train",
            image_dir / "test",
            image_dir / "train" / "images",
            image_dir / "test" / "images",
        ]
        for folder in ("Normal", "COVID", "Lung_Opacity", "Viral Pneumonia", "Viral_Pneumonia"):
            directories += [image_dir / folder, image_dir / folder / "images"]
    elif kind == "tb":
        for folder in ("Tuberculosis", "Normal", "tuberculosis", "normal"):
            directories += [image_dir / folder, image_dir / folder / "images"]

    for variant in variants:
        for directory in directories:
            candidate = directory / variant
            if candidate.is_file():
                return str(candidate)

    if cache.get("map") is None:
        cache["map"] = build_recursive_file_map(str(image_dir))
    for variant in variants:
        resolved = cache["map"].get(variant)
        if resolved:
            return resolved
    return None


def multi_hot(text):
    labels = torch.zeros(len(NIH_PATHOLOGY_LIST), dtype=torch.float32)
    text = str(text)
    for index, pathology in enumerate(NIH_PATHOLOGY_LIST):
        if pathology in text:
            labels[index] = 1.0
    return labels


def covid_label(raw):
    token = str(raw).strip().lower().replace("_", " ").replace("-", " ")
    if "non" in token and "covid" in token:
        return 1
    if "covid" in token:
        return 2
    if any(key in token for key in ("pneumonia", "opacity", "viral", "bacterial")):
        return 1
    return 0


def assert_patient_isolation(df, name):
    available = [split for split in ("train", "val", "test") if split in set(df["split"].astype(str))]
    groups = {
        split: set(df.loc[df["split"].astype(str) == split, "patient_id"].dropna().astype(str))
        for split in available
    }
    for i, left in enumerate(available):
        for right in available[i + 1:]:
            overlap = groups[left] & groups[right]
            assert not overlap, f"[CRITICAL] {name} patient leakage between {left}/{right}: {len(overlap)} IDs"
    print(f"[PASS] {name}: zero patient leakage across {available}.")


def carve_val_by_patient(df, val_ratio=0.10):
    df = df.copy()
    train_mask = df["split"].astype(str) == "train"
    train_df = df[train_mask]
    if train_df.empty or "patient_id" not in df.columns:
        return df
    patient = train_df.groupby("patient_id").agg(
        label=("label", lambda values: int(values.mode().iloc[0])),
        source=("data_source", lambda values: str(values.mode().iloc[0])),
    )
    patient["stratum"] = patient["label"].astype(str) + "::" + patient["source"].astype(str)
    patient_ids = patient.index.to_numpy()
    n_val = max(1, int(round(len(patient_ids) * val_ratio)))
    n_val = min(n_val, len(patient_ids) - 1) if len(patient_ids) > 1 else 0
    if n_val > 0:
        try:
            _, val_ids = train_test_split(
                patient_ids, test_size=n_val, random_state=SEED,
                stratify=patient["stratum"].to_numpy(),
            )
        except ValueError:
            _, val_ids = train_test_split(patient_ids, test_size=n_val, random_state=SEED)
        df.loc[train_mask & df["patient_id"].isin(val_ids), "split"] = "val"
    return df


def load_nih_dataframe(paths):
    nih_dir = paths.get("nih_dir")
    if not nih_dir or not Path(nih_dir).is_dir():
        return None, "NIH mount missing"
    split_csv = paths.get("nih_split_csv") or locate_file(
        "patient_splits.csv", [nih_dir, paths.get("split_dir"), REPO / "dataset" / "splits"]
    )
    df = None
    if split_csv and Path(split_csv).is_file():
        df = pd.read_csv(split_csv)
        source = f"patient_splits.csv ({split_csv})"
    else:
        metadata = locate_file("Data_Entry_2017.csv", [nih_dir]) or locate_file(
            "Data_Entry_2017_v2020.csv", [nih_dir]
        )
        if not metadata:
            return None, "no patient_splits.csv or Data_Entry_2017.csv"
        raw = pd.read_csv(metadata)
        df = split_nih_by_patient_id(
            raw, patient_col="Patient ID", train_ratio=0.70,
            val_ratio=0.10, test_ratio=0.20, seed=SEED,
        )
        source = f"on-the-fly Patient_ID split from {metadata}"
    if "Image Index" not in df.columns:
        for candidate in ("filename", "image_id"):
            if candidate in df.columns:
                df["Image Index"] = df[candidate]
                break
    if "Finding Labels" not in df.columns and "label" in df.columns:
        df["Finding Labels"] = df["label"].astype(str)
    if "Image Index" not in df.columns or "Finding Labels" not in df.columns or "split" not in df.columns:
        return None, "NIH dataframe is missing Image Index / Finding Labels / split"
    if "Patient ID" in df.columns:
        df["patient_id"] = df["Patient ID"].astype(str)
        assert_patient_isolation(df, "NIH")
    return df, source


def find_class_dir(root, class_names, max_depth=4):
    root = Path(root)
    base = len(root.parts)
    for dirpath, dirnames, _ in os.walk(root):
        here = Path(dirpath)
        if len(here.parts) - base > max_depth:
            dirnames[:] = []
            continue
        for name in dirnames:
            if name.lower() in class_names:
                return here / name
    return None


def load_tb_dataframe(paths):
    tb_dir = paths.get("tb_dir")
    split_csv = paths.get("tb_split_csv") or locate_file(
        "tb_splits.csv", [tb_dir, paths.get("split_dir"), REPO / "dataset" / "splits"]
    )
    if split_csv and Path(split_csv).is_file():
        df = pd.read_csv(split_csv)
        source = f"tb_splits.csv ({split_csv})"
        if "path" not in df.columns and (not tb_dir or not Path(tb_dir).is_dir()):
            return None, "tb_splits.csv found but the TB image mount is missing"
    else:
        if not tb_dir or not Path(tb_dir).is_dir():
            return None, "TB mount missing"
        records = []
        for label, names in ((1, {"tuberculosis"}), (0, {"normal"})):
            class_dir = find_class_dir(tb_dir, names)
            if class_dir is None:
                continue
            for path in iter_images(class_dir, max_depth=4, limit=30000):
                records.append({"filename": path.name, "path": str(path), "label": float(label)})
        if not records:
            return None, "no Normal/Tuberculosis class folders under the TB mount"
        df = pd.DataFrame(records)
        train_df, temp_df = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df["label"])
        val_df, test_df = train_test_split(
            temp_df, test_size=0.6667, random_state=SEED, stratify=temp_df["label"]
        )
        df = pd.concat(
            [train_df.assign(split="train"), val_df.assign(split="val"), test_df.assign(split="test")],
            ignore_index=True,
        )
        source = "stratified Normal/Tuberculosis folder scan"
    if "filename" not in df.columns:
        for candidate in ("Image Index", "image_id"):
            if candidate in df.columns:
                df["filename"] = df[candidate]
                break
    if "label" not in df.columns and "diagnosis" in df.columns:
        df["label"] = df["diagnosis"].astype(str).str.lower().str.contains("tuberculosis").astype(float)
    if "filename" not in df.columns or "label" not in df.columns or "split" not in df.columns:
        return None, "TB dataframe is missing filename / label / split"
    return df, source


def load_covid_dataframe(paths):
    covid_dir = paths.get("covid_dir")
    if not covid_dir or not Path(covid_dir).is_dir():
        return None, "COVID mount missing"
    train_txt, test_txt = find_covidx_manifests(str(covid_dir))
    if train_txt and test_txt:
        records = []
        for txt_path, split in ((train_txt, "train"), (test_txt, "test")):
            with open(txt_path, "r", encoding="utf-8", errors="ignore") as handle:
                for line in handle:
                    parts = line.strip().split()
                    if len(parts) < 2 or parts[0].startswith("#"):
                        continue
                    if len(parts) >= 4:
                        patient_id, filename, raw_class, source = parts[0], parts[1], parts[2], parts[3]
                    elif len(parts) >= 3:
                        patient_id, filename, raw_class = parts[0], parts[1], parts[2]
                        source = "covidx"
                    else:
                        filename, raw_class = parts[0], parts[1]
                        patient_id = Path(filename).stem
                        source = "covidx"
                    records.append(
                        {
                            "patient_id": str(patient_id),
                            "filename": Path(filename).name,
                            "label": covid_label(raw_class),
                            "label3": covid_label(raw_class),
                            "data_source": str(source),
                            "split": split,
                        }
                    )
        df = pd.DataFrame(records).drop_duplicates(subset=["filename"], keep="first")
        train_patients = set(df.loc[df["split"] == "train", "patient_id"])
        test_patients = set(df.loc[df["split"] == "test", "patient_id"])
        leaked = train_patients & test_patients
        if leaked:
            print(f"[!] Removing {len(leaked)} patients leaking between official COVIDx train/test.")
            df = df[~((df["split"] == "train") & df["patient_id"].isin(leaked))]
        df = carve_val_by_patient(df, val_ratio=0.10)
        assert_patient_isolation(df, "COVIDx CXR-4")
        return df, f"COVIDx manifests ({train_txt})"

    split_csv = paths.get("covid_split_csv") or locate_file(
        "covid_splits.csv", [covid_dir, paths.get("split_dir"), REPO / "dataset" / "splits"]
    )
    if split_csv and Path(split_csv).is_file():
        df = pd.read_csv(split_csv)
        if "filename" not in df.columns:
            for candidate in ("Image Index", "image_id"):
                if candidate in df.columns:
                    df["filename"] = df[candidate]
                    break
        if "patient_id" not in df.columns:
            df["patient_id"] = df["filename"].astype(str).map(lambda name: Path(name).stem)
        if "class_name" in df.columns:
            def map_class(name):
                token = str(name).lower()
                if "covid" in token:
                    return 2
                if "opacity" in token or "pneumonia" in token or "viral" in token:
                    return 1
                return 0
            df["label3"] = df["class_name"].map(map_class)
        elif "label" in df.columns:
            raw_labels = pd.to_numeric(df["label"], errors="coerce")
            if raw_labels.notna().all() and float(raw_labels.max()) <= 1.0:
                df["label3"] = raw_labels.astype(int).map({0: 0, 1: 2})
            elif raw_labels.notna().all():
                df["label3"] = raw_labels.astype(int).map({0: 2, 1: 0, 2: 1, 3: 1}).fillna(0).astype(int)
            else:
                df["label3"] = df["label"].map(covid_label)
        else:
            return None, "covid_splits.csv has no class_name/label column"
        df["label"] = df["label3"]
        if "data_source" not in df.columns:
            df["data_source"] = df.get("source", "legacy")
        if "split" not in df.columns:
            return None, "covid_splits.csv has no split column"
        if df["patient_id"].nunique() > 0.95 * len(df):
            print("[!] COVID patient_id is almost unique per image; the legacy Radiography CSV cannot enforce patient-level isolation. Prefer COVIDx CXR-4 manifests for Rule 3.")
        assert_patient_isolation(df, "COVID-19 Radiography (source-stratified CSV)")
        return df, f"covid_splits.csv ({split_csv})"

    return None, "no COVIDx manifests nor source-stratified covid_splits.csv (Rule 3 guard)"


def records_from_dataframe(df, split, image_dir, kind, limit=None):
    records = []
    cache = {"map": None}
    subset = df[df["split"].astype(str) == split]
    if limit and len(subset) > limit:
        subset = subset.sample(n=limit, random_state=SEED)
    for _, row in subset.iterrows():
        name = row.get("filename")
        if name is None or (isinstance(name, float) and np.isnan(name)):
            name = row.get("Image Index")
        if name is None or (isinstance(name, float) and np.isnan(name)):
            continue
        path = row.get("path")
        if not isinstance(path, str) or not Path(path).is_file():
            path = resolve_image(image_dir, name, kind, cache)
        if not path:
            continue
        if kind == "nih":
            labels = multi_hot(row.get("Finding Labels", ""))
            task_id = 0
        elif kind == "tb":
            value = row.get("label", 0.0)
            if pd.isna(value):
                value = 0.0
            labels = torch.tensor([float(value)], dtype=torch.float32)
            task_id = 1
        else:
            value = row.get("label3", 0)
            if pd.isna(value):
                value = 0
            labels = torch.tensor(int(value), dtype=torch.long)
            task_id = 2
        records.append(
            {"path": str(path), "labels": labels, "task_id": task_id, "image_id": Path(str(path)).name}
        )
    return records


def synthetic_records(task_id, count):
    records = []
    for index in range(count):
        if task_id == 0:
            labels = torch.zeros(len(NIH_PATHOLOGY_LIST), dtype=torch.float32)
            labels[index % len(NIH_PATHOLOGY_LIST)] = 1.0
            if index % 3 == 0:
                labels[(index * 5 + 3) % len(NIH_PATHOLOGY_LIST)] = 1.0
        elif task_id == 1:
            labels = torch.tensor([1.0 if index % 4 == 0 else 0.0], dtype=torch.float32)
        else:
            labels = torch.tensor(index % 3, dtype=torch.long)
        records.append(
            {"path": None, "labels": labels, "task_id": task_id, "image_id": f"synthetic_{task_id}_{index}"}
        )
    return records


def describe_frame(name, df, label_column="label"):
    if df is None:
        print(f"[*] {name}: unavailable")
        return
    splits = df["split"].astype(str).value_counts().to_dict()
    print(f"[*] {name}: rows={len(df)} | splits={splits}")
    if label_column in df.columns:
        counts = df[label_column].value_counts(dropna=False)
        print(f"    {label_column} distribution (top 12 of {len(counts)} unique): {counts.head(12).to_dict()}")


print("[*] Resolving dataset mounts under /kaggle/input ...")
try:
    PATHS = resolve_dataset_paths()
except Exception as exc:
    PATHS = {}
    print(f"[!] resolve_dataset_paths failed ({exc}); synthetic mode will be used.")
print("[*] Mount status:", PATHS.get("missing", "no mount") if PATHS else "no mount")
print("[*] Resolved paths:", {key: PATHS.get(key) for key in ("nih_dir", "covid_dir", "tb_dir", "split_dir")})

NIH_DF, NIH_SOURCE = load_nih_dataframe(PATHS)
COVID_DF, COVID_SOURCE = load_covid_dataframe(PATHS)
TB_DF, TB_SOURCE = load_tb_dataframe(PATHS)
print(f"[*] NIH   : {NIH_SOURCE}")
print(f"[*] COVID : {COVID_SOURCE}")
print(f"[*] TB    : {TB_SOURCE}")
describe_frame("NIH dataframe", NIH_DF, "Finding Labels")
describe_frame("COVID dataframe", COVID_DF, "label")
describe_frame("TB dataframe", TB_DF, "label")

## 2. Balanced epoch (60 : 25 : 15) and task-isolated DataLoaders

`BalancedMultiTaskSampler` guarantees a fixed per-batch composition derived from `TASK_RATIOS` and `BATCH_SIZE` (12 : 5 : 3 at batch size 20), preventing task starvation. The number of batches per epoch is `SAMPLES_PER_EPOCH // BATCH_SIZE`, so the exact image counts are deterministic and printed at runtime.

- `fast`: 3,000 samples/epoch
- `deep` (default): 6,000 samples/epoch
- `full`: 10,000 samples/epoch

The sampler cycles each task pool with fresh permutations, so no task is ever starved even when a pool is smaller than its per-epoch quota.

In [ ]:
# ---------------------------------------------------------------------------
# 2. Balanced sampler, medical preprocessing and task-isolated DataLoaders
# ---------------------------------------------------------------------------
def letterbox_gray(image, size):
    h, w = image.shape[:2]
    scale = size / max(h, w)
    nh = max(1, int(round(h * scale)))
    nw = max(1, int(round(w * scale)))
    interp = cv2.INTER_AREA if scale < 1.0 else cv2.INTER_LINEAR
    resized = cv2.resize(image, (nw, nh), interpolation=interp)
    top = (size - nh) // 2
    left = (size - nw) // 2
    return cv2.copyMakeBorder(
        resized, top, size - nh - top, left, size - nw - left,
        cv2.BORDER_CONSTANT, value=0,
    )


def clahe_contrast_jitter(image_u8, rng, p=0.50):
    if rng.random() > p:
        return image_u8
    clip_limit = float(rng.uniform(1.5, 3.5))
    tile = int(rng.choice([4, 8, 16]))
    return cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(tile, tile)).apply(image_u8)


class MultiTaskRecordDataset(Dataset):
    def __init__(self, records, img_size=256, training=True, seed=SEED):
        self.records = records
        self.img_size = int(img_size)
        self.training = bool(training)
        self.seed = int(seed)

    def __len__(self):
        return len(self.records)

    def _synthetic_image(self, index):
        rng = np.random.default_rng(self.seed + index * 104729)
        h = w = self.img_size
        base = int(rng.integers(20, 60))
        image = np.full((h, w), base, dtype=np.uint8)
        cv2.ellipse(image, (w // 2, h // 2), (int(w * 0.40), int(h * 0.45)), 0, 0, 360, base + 90, -1)
        for cx in (0.32, 0.68):
            cv2.ellipse(image, (int(w * cx), int(h * 0.48)), (int(w * 0.12), int(h * 0.26)), 0, 0, 360, base + 130, -1)
        return cv2.add(image, rng.integers(0, 18, (h, w), dtype=np.uint8))

    def __getitem__(self, index):
        record = self.records[index]
        rng = np.random.default_rng(self.seed + index * 7919 + (1 if self.training else 0))
        image = None
        if record["path"]:
            image = cv2.imread(record["path"], cv2.IMREAD_GRAYSCALE)
        if image is None:
            image = self._synthetic_image(index)
        image = letterbox_gray(image, self.img_size)
        if self.training:
            if rng.random() < 0.5:
                image = cv2.flip(image, 1)
            image = clahe_contrast_jitter(image, rng, p=0.5)
        rgb = np.stack([image, image, image], axis=-1).astype(np.float32) / 255.0
        rgb = (rgb - IMAGENET_MEAN) / IMAGENET_STD
        return {
            "image": torch.from_numpy(rgb.transpose(2, 0, 1)),
            "labels": record["labels"],
            "task_id": torch.tensor(record["task_id"], dtype=torch.long),
            "image_id": record.get("image_id", ""),
        }


class BalancedMultiTaskSampler(Sampler):
    def __init__(self, nih_indices, covid_indices, tb_indices, batch_size=20,
                 batches_per_epoch=150, ratios=(0.60, 0.25, 0.15), seed=SEED):
        self.pools = [np.asarray(nih_indices), np.asarray(covid_indices), np.asarray(tb_indices)]
        if any(len(pool) == 0 for pool in self.pools):
            raise ValueError("All three task pools must be non-empty.")
        quotas = np.asarray(ratios, dtype=float)
        quotas = quotas / quotas.sum() * batch_size
        counts = np.floor(quotas).astype(int)
        remainder = batch_size - int(counts.sum())
        if remainder > 0:
            counts[np.argsort(-(quotas - counts))[:remainder]] += 1
        self.batch_counts = tuple(int(value) for value in counts)
        self.batch_size = int(batch_size)
        self.num_batches = int(batches_per_epoch)
        self.seed = int(seed)
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return self.num_batches

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        pools = [pool.copy() for pool in self.pools]
        pointers = [0, 0, 0]
        for _ in range(self.num_batches):
            batch = []
            for task_index, need in enumerate(self.batch_counts):
                while need > 0:
                    if pointers[task_index] >= len(pools[task_index]):
                        pools[task_index] = rng.permutation(self.pools[task_index])
                        pointers[task_index] = 0
                    take = min(need, len(pools[task_index]) - pointers[task_index])
                    batch.extend(pools[task_index][pointers[task_index]:pointers[task_index] + take].tolist())
                    pointers[task_index] += take
                    need -= take
            rng.shuffle(batch)
            yield batch


def worker_init(worker_id):
    cv2.setNumThreads(0)
    torch.set_num_threads(1)


nih_train = records_from_dataframe(NIH_DF, "train", PATHS.get("nih_dir"), "nih", limit=POOL_LIMITS["nih"]) if NIH_DF is not None else []
covid_train = records_from_dataframe(COVID_DF, "train", PATHS.get("covid_dir"), "covid", limit=POOL_LIMITS["covid"]) if COVID_DF is not None else []
tb_train = records_from_dataframe(TB_DF, "train", PATHS.get("tb_dir"), "tb", limit=POOL_LIMITS["tb"]) if TB_DF is not None else []

nih_val = records_from_dataframe(NIH_DF, "val", PATHS.get("nih_dir"), "nih", limit=3000) if NIH_DF is not None else []
covid_val = records_from_dataframe(COVID_DF, "val", PATHS.get("covid_dir"), "covid", limit=2000) if COVID_DF is not None else []
tb_val = records_from_dataframe(TB_DF, "val", PATHS.get("tb_dir"), "tb", limit=2000) if TB_DF is not None else []

nih_test = records_from_dataframe(NIH_DF, "test", PATHS.get("nih_dir"), "nih", limit=3000) if NIH_DF is not None else []
covid_test = records_from_dataframe(COVID_DF, "test", PATHS.get("covid_dir"), "covid", limit=2000) if COVID_DF is not None else []
tb_test = records_from_dataframe(TB_DF, "test", PATHS.get("tb_dir"), "tb", limit=2000) if TB_DF is not None else []

if NIH_DF is not None and not nih_train:
    print("[!] NIH split exists but no images resolved. Check that the mounted dataset contains the filenames in patient_splits.csv.")
if COVID_DF is not None and not covid_train:
    print("[!] COVID split exists but no images resolved. Check the COVIDx manifest filenames.")
if TB_DF is not None and not tb_train:
    print("[!] TB split exists but no images resolved. Mount 'tuberculosis-tb-chest-xray-dataset' (Normal/Tuberculosis), not only the Stage 1 mask dataset 'montgo-shenzhen-resize'.")

data_modes = []
if not nih_train:
    nih_train = synthetic_records(0, 1800)
    nih_val = synthetic_records(0, 300)
    nih_test = []
    data_modes.append("NIH=SYNTHETIC")
else:
    data_modes.append("NIH=REAL")
if not covid_train:
    covid_train = synthetic_records(2, 750)
    covid_val = synthetic_records(2, 150)
    covid_test = []
    data_modes.append("COVID=SYNTHETIC")
else:
    data_modes.append("COVID=REAL")
if not tb_train:
    tb_train = synthetic_records(1, 450)
    tb_val = synthetic_records(1, 150)
    tb_test = []
    data_modes.append("TB=SYNTHETIC")
else:
    data_modes.append("TB=REAL")
if not nih_val:
    nih_val = synthetic_records(0, 300)
if not covid_val:
    covid_val = synthetic_records(2, 150)
if not tb_val:
    tb_val = synthetic_records(1, 150)
REAL_HEADS = {
    "A": "NIH=REAL" in data_modes,
    "B": "TB=REAL" in data_modes,
    "C": "COVID=REAL" in data_modes,
}
print("[*] Data modes:", " | ".join(data_modes))
print(f"[*] Train pools: NIH={len(nih_train)} | COVID={len(covid_train)} | TB={len(tb_train)}")
print(f"[*] Val pools  : NIH={len(nih_val)} | COVID={len(covid_val)} | TB={len(tb_val)}")
print(f"[*] Test pools : NIH={len(nih_test)} | COVID={len(covid_test)} | TB={len(tb_test)}")

train_records = nih_train + covid_train + tb_train
nih_indices = list(range(0, len(nih_train)))
covid_indices = list(range(len(nih_train), len(nih_train) + len(covid_train)))
tb_indices = list(range(len(nih_train) + len(covid_train), len(train_records)))

sampler = BalancedMultiTaskSampler(
    nih_indices, covid_indices, tb_indices,
    batch_size=BATCH_SIZE, batches_per_epoch=BATCHES_PER_EPOCH,
    ratios=TASK_RATIOS, seed=SEED,
)
actual = tuple(int(count) * len(sampler) for count in sampler.batch_counts)
total_samples = sum(actual)
for got, target in zip(actual, TASK_RATIOS):
    assert abs(got / total_samples - target) < 0.03, f"[Rule 5] Task ratio violation: {actual} (target: {TASK_RATIOS})"
print(f"[Rule 5] Batch composition: {sampler.batch_counts} (NIH:COVID:TB) for batch_size={BATCH_SIZE}")
print(f"[Rule 5] Per epoch: NIH={actual[0]} | COVID={actual[1]} | TB={actual[2]} | total={total_samples} over {len(sampler)} batches")
print("[Rule 4] Bounding boxes: NOT LOADED. Classification and WSOL supervision use image-level labels only.")

covid_counts = np.zeros(NUM_CLASSES_C, dtype=np.float64)
for record in covid_train:
    if record["task_id"] == 2:
        covid_counts[int(record["labels"].item())] += 1
if covid_counts.sum() > 0:
    ce_weights = covid_counts.sum() / (NUM_CLASSES_C * np.clip(covid_counts, 1, None))
    ce_weights = ce_weights / ce_weights.mean()
    ce_weights = np.clip(ce_weights, 0.2, 5.0)
else:
    ce_weights = np.ones(NUM_CLASSES_C, dtype=np.float64)

tb_pos = sum(int(record["labels"].item()) for record in tb_train if record["task_id"] == 1)
tb_neg = len(tb_train) - tb_pos
tb_pos_weight = float(np.clip(tb_neg / max(1, tb_pos), 1.0, 10.0))
print("[*] Head C CE weights:", np.round(ce_weights, 3).tolist(), "| counts:", covid_counts.astype(int).tolist())
print("[*] Head B pos_weight:", round(tb_pos_weight, 3), f"| pos={tb_pos} neg={tb_neg}")

train_dataset = MultiTaskRecordDataset(train_records, img_size=IMG_SIZE, training=True, seed=SEED)
loader_kwargs = {
    "num_workers": NUM_WORKERS,
    "pin_memory": device.type == "cuda",
    "persistent_workers": NUM_WORKERS > 0,
    "worker_init_fn": worker_init if NUM_WORKERS > 0 else None,
}


def make_eval_loader(records):
    if not records:
        return None
    return DataLoader(
        MultiTaskRecordDataset(records, IMG_SIZE, False, SEED),
        batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=multitask_collate_fn, **loader_kwargs,
    )


train_loader = DataLoader(train_dataset, batch_sampler=sampler, collate_fn=multitask_collate_fn, **loader_kwargs)
val_loaders = {
    "head_a": make_eval_loader(nih_val),
    "head_b": make_eval_loader(tb_val),
    "head_c": make_eval_loader(covid_val),
}
test_loaders = {
    "head_a": make_eval_loader(nih_test),
    "head_b": make_eval_loader(tb_test),
    "head_c": make_eval_loader(covid_test),
}
print(f"[*] Train batches/epoch: {len(train_loader)} | Val batches: " + ", ".join(f"{k}={len(v) if v is not None else 0}" for k, v in val_loaders.items()))
print(f"[*] Test batches: " + ", ".join(f"{k}={len(v) if v is not None else 0}" for k, v in test_loaders.items()))

In [ ]:
# ---------------------------------------------------------------------------
# 3. AG-MTCNet model, frozen anatomical gate, masked loss and evaluation helpers
# ---------------------------------------------------------------------------
ckpt_dir = REPO / "checkpoints"
ckpt_dir.mkdir(parents=True, exist_ok=True)


def load_anatomical_gate():
    candidates = [
        Path("/kaggle/input/stage1-dense-unet/dense_unet_best.pth"),
        Path("/kaggle/input/stage1-dense-unet/dense_unet.pth"),
        REPO / "checkpoints" / "dense_unet_best.pth",
        REPO / "checkpoints" / "dense_unet.pth",
        Path("/kaggle/working") / "dense_unet_best.pth",
        Path("/kaggle/working") / "dense_unet.pth",
    ]
    candidates += [Path(path) for path in find_input_files("dense_unet")]
    seen = set()
    for checkpoint_path in candidates:
        checkpoint_path = Path(checkpoint_path)
        if str(checkpoint_path) in seen or not checkpoint_path.is_file():
            continue
        seen.add(str(checkpoint_path))
        try:
            payload = safe_torch_load(checkpoint_path)
            state_dict = payload.get("model_state_dict", payload) if isinstance(payload, dict) else payload
            gate = DenseUNet(pretrained=False).to(device)
            missing, unexpected = gate.load_state_dict(state_dict, strict=False)
            loaded = len(state_dict) - len(unexpected)
            if loaded < 0.95 * len(state_dict):
                raise RuntimeError(f"Gate checkpoint incompatible: {loaded}/{len(state_dict)} keys loaded")
            gate.eval()
            for parameter in gate.parameters():
                parameter.requires_grad = False
            return gate, str(checkpoint_path)
        except Exception as exc:
            print(f"[!] Gate checkpoint rejected {checkpoint_path}: {exc}")
    return None, None


def load_converged_base_weights(model):
    candidates = [
        REPO / "checkpoints" / "best_model.pth",
        Path("/kaggle/input/best-model/best_model.pth"),
        Path("/kaggle/input/lungray-checkpoints/best_model.pth"),
        Path("/kaggle/working/checkpoints/best_model.pth"),
        Path("/kaggle/working/best_model.pth"),
    ]
    candidates += [Path(path) for path in find_input_files("best_model")]
    if RESUME:
        candidates.append(Path("/kaggle/working/ag_mtcnet_best.pth"))
        candidates.append(REPO / "checkpoints" / "ag_mtcnet_best.pth")
    seen = set()
    model_state = model.state_dict()
    for checkpoint_path in candidates:
        checkpoint_path = Path(checkpoint_path)
        if str(checkpoint_path) in seen or not checkpoint_path.is_file():
            continue
        seen.add(str(checkpoint_path))
        try:
            state = safe_torch_load(checkpoint_path)
            if isinstance(state, dict):
                state = state.get("model_state_dict", state.get("state_dict", state))
            if not isinstance(state, dict):
                raise TypeError("checkpoint payload is not a state dict")
            filtered = {
                key: value for key, value in state.items()
                if key in model_state and hasattr(value, "shape") and model_state[key].shape == value.shape
            }
            if not filtered:
                raise RuntimeError("no shape-compatible tensors")
            model.load_state_dict(filtered, strict=False)
            print(f"[OK] Warm-started {len(filtered)}/{len(model_state)} tensors from {checkpoint_path}.")
            return True
        except Exception as exc:
            print(f"[!] Warm-start rejected {checkpoint_path}: {exc}")
    print("[*] No compatible warm-start checkpoint found.")
    return False


def build_model(pretrained):
    return MultiTaskChestXrayNet(
        backbone_name="densenet121",
        pretrained=pretrained,
        img_size=IMG_SIZE,
        dropout_rate=0.20,
        head_norm="layernorm",
        num_classes_c=NUM_CLASSES_C,
    ).to(device)


gate_model, gate_source = load_anatomical_gate()
if gate_model is None:
    message = "[!] No Stage 1 anatomical gate found: training proceeds ungated."
    print(message)
    if REQUIRE_GATE:
        raise FileNotFoundError(message)
else:
    print(f"[OK] Frozen anatomical gate loaded from {gate_source} (requires_grad=False).")

model = build_model(pretrained=False)
warmstarted = load_converged_base_weights(model)
if not warmstarted and USE_IMAGENET:
    print("[*] Rebuilding the encoder with ImageNet weights because no warm-start was found.")
    model = build_model(pretrained=True)
print(f"[*] AG-MTCNet instantiated: {sum(p.numel() for p in model.parameters()):,} parameters (Head C = {NUM_CLASSES_C} classes).")

asl_reference = AsymmetricFocalLoss(gamma_neg=4.0, gamma_pos=0.0, clip=0.05)
criterion = MaskedMultiTaskLoss(
    weights={"head_a": 1.0, "head_b": 0.8, "head_c": 0.8},
    head_b_pos_weight=tb_pos_weight,
    head_a_criterion="asl",
    asl_gamma_neg=4.0,
    asl_gamma_pos=0.0,
    asl_clip=0.05,
    head_c_criterion="ce",
    ce_weight_c=ce_weights.tolist(),
).to(device)
print("[*] ASL config: gamma_pos=0, gamma_neg=4, clip=0.05 (rare Hernia/Fibrosis gradient rescue).")
print("[*] Masked loss: Head A ASL | Head B weighted BCEWithLogits | Head C class-weighted CrossEntropy.")
print(f"[*] Head C class weights: {np.round(ce_weights, 3).tolist()} | Head B pos_weight: {tb_pos_weight:.3f}")


def build_param_groups(model, backbone_lr=1e-5, head_lr=1e-4, weight_decay=1e-4, decay=0.85):
    layer_order = [
        "norm5", "denseblock4", "transition3", "denseblock3", "transition2",
        "denseblock2", "transition1", "denseblock1", "norm0", "conv0",
    ]
    grouped = {}
    for name, parameter in model.backbone.named_parameters():
        if not parameter.requires_grad:
            continue
        depth = len(layer_order) - 1
        for index, block in enumerate(layer_order):
            if block in name:
                depth = index
                break
        learning_rate = backbone_lr * (decay ** depth)
        grouped.setdefault(round(learning_rate, 12), []).append(parameter)
    groups = [
        {"params": parameters, "lr": learning_rate, "weight_decay": weight_decay}
        for learning_rate, parameters in sorted(grouped.items(), reverse=True)
    ]
    head_parameters = [
        parameter for name, parameter in model.named_parameters()
        if parameter.requires_grad and not name.startswith("backbone.")
    ]
    groups.append({"params": head_parameters, "lr": head_lr, "weight_decay": weight_decay})
    return groups


class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.decay = float(decay)
        self.ema = {}
        self.backup = {}
        self.update(model, init=True)

    @torch.no_grad()
    def update(self, model, init=False):
        for key, value in model.state_dict().items():
            if not value.is_floating_point() or value.dim() == 0:
                continue
            if key not in self.ema:
                self.ema[key] = value.detach().clone().float()
            elif not init:
                self.ema[key].mul_(self.decay).add_(value.detach().float(), alpha=1.0 - self.decay)

    @torch.no_grad()
    def store(self, model):
        self.backup = {key: value.detach().clone() for key, value in model.state_dict().items()}

    @torch.no_grad()
    def copy_to(self, model):
        state = model.state_dict()
        for key, value in self.ema.items():
            if key in state:
                state[key].copy_(value.to(device=state[key].device, dtype=state[key].dtype))

    @torch.no_grad()
    def restore(self, model):
        model.load_state_dict(self.backup, strict=True)


def safe_macro_auc(y_true, y_prob):
    scores = []
    for class_index in range(y_true.shape[1]):
        column = y_true[:, class_index]
        if len(np.unique(column)) > 1:
            scores.append(roc_auc_score(column, y_prob[:, class_index]))
    return float(np.mean(scores)) if scores else float("nan")


def safe_macro_ovr(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int).reshape(-1)
    scores = []
    for class_index in np.unique(y_true):
        binary = (y_true == class_index).astype(int)
        if len(np.unique(binary)) > 1:
            scores.append(roc_auc_score(binary, y_prob[:, int(class_index)]))
    return float(np.mean(scores)) if scores else float("nan")


@torch.no_grad()
def collect_predictions(loader, head, max_batches=MAX_VAL_BATCHES):
    if loader is None:
        return np.zeros((0,)), np.zeros((0,))
    model.eval()
    probs, targets = [], []
    for step, batch in enumerate(loader):
        if max_batches is not None and step >= max_batches:
            break
        images = batch["images"].to(device, non_blocking=True)
        if gate_model is not None:
            lung_masks = gate_model(images)
            images = apply_dilated_soft_mask_tensor(images, lung_masks)
        logits = model(images)
        if head == "head_a":
            probs.append(torch.sigmoid(logits["head_a"]).float().cpu().numpy())
        elif head == "head_b":
            probs.append(torch.sigmoid(logits["head_b"]).float().cpu().numpy().reshape(-1))
        else:
            probs.append(torch.softmax(logits["head_c"], dim=-1).float().cpu().numpy())
        targets.append(torch.stack([target.detach().cpu() for target in batch["labels"]]))
    if not probs:
        return np.zeros((0,)), np.zeros((0,))
    return torch.cat(targets, dim=0).numpy(), np.concatenate(probs, axis=0)


def evaluate_heads(loaders, max_batches=MAX_VAL_BATCHES, return_predictions=False):
    y_a, p_a = collect_predictions(loaders.get("head_a"), "head_a", max_batches)
    y_b, p_b = collect_predictions(loaders.get("head_b"), "head_b", max_batches)
    y_c, p_c = collect_predictions(loaders.get("head_c"), "head_c", max_batches)

    if np.asarray(y_a).size:
        y_a = np.asarray(y_a).reshape(-1, len(NIH_PATHOLOGY_LIST))
        p_a = np.asarray(p_a)
        auc_a = safe_macro_auc(y_a, p_a)
    else:
        auc_a = float("nan")
    if np.asarray(y_b).size:
        y_b = np.asarray(y_b).reshape(-1)
        p_b = np.asarray(p_b).reshape(-1)
        auc_b = float(roc_auc_score(y_b, p_b)) if len(np.unique(y_b)) > 1 else float("nan")
    else:
        auc_b = float("nan")
    if np.asarray(y_c).size:
        y_c = np.asarray(y_c).reshape(-1).astype(int)
        p_c = np.asarray(p_c)
        accuracy_c = float(accuracy_score(y_c, p_c.argmax(axis=1)))
        auc_c = safe_macro_ovr(y_c, p_c)
    else:
        accuracy_c = float("nan")
        auc_c = float("nan")

    parts = []
    if REAL_HEADS["A"] and np.isfinite(auc_a):
        parts.append((0.6, auc_a))
    if REAL_HEADS["B"] and np.isfinite(auc_b):
        parts.append((0.2, auc_b))
    if REAL_HEADS["C"] and np.isfinite(auc_c):
        parts.append((0.2, auc_c))
    if not parts:
        for weight, value in ((0.6, auc_a), (0.2, auc_b), (0.2, auc_c)):
            if np.isfinite(value):
                parts.append((weight, value))
    score = float(sum(weight * value for weight, value in parts) / sum(weight for weight, _ in parts)) if parts else 0.0
    metrics = {
        "auc_a": float(auc_a),
        "auc_b": float(auc_b),
        "auc_c": float(auc_c),
        "acc_c": float(accuracy_c),
        "score": score,
    }
    if return_predictions:
        return metrics, {"a": (y_a, p_a), "b": (y_b, p_b), "c": (y_c, p_c)}
    return metrics


def tune_youden(y_true, y_prob, names):
    thresholds = {}
    for index in range(y_true.shape[1]):
        column = y_true[:, index]
        if len(np.unique(column)) < 2:
            continue
        fpr, tpr, threshold_values = roc_curve(column, y_prob[:, index])
        j_statistic = tpr - fpr
        best = int(np.argmax(j_statistic))
        thresholds[names[index]] = float(np.clip(threshold_values[best], 0.0, 1.0))
    return thresholds


scaler = build_grad_scaler()
ema = ModelEMA(model, EMA_DECAY) if USE_EMA else None
history = []
print(f"[*] Gradient scaler: {scaler.__class__.__name__} | EMA: {USE_EMA}")

In [ ]:
# ---------------------------------------------------------------------------
# 4. Two-stage fine-tuning with validation-selected best checkpoint
# ---------------------------------------------------------------------------
def train_epoch(model, gate, dataloader, optimizer, scaler, epoch_name, ema=None,
                max_batches=None, grad_accum_steps=GRAD_ACCUM_STEPS):
    model.train()
    backbone = getattr(model, "backbone", None)
    if backbone is not None and not any(parameter.requires_grad for parameter in backbone.parameters()):
        backbone.eval()
    if gate is not None:
        gate.eval()
    running_total, count = 0.0, 0
    running_heads = {"loss_a": 0.0, "loss_b": 0.0, "loss_c": 0.0}
    optimizer.zero_grad(set_to_none=True)
    for step, batch in enumerate(dataloader):
        if max_batches is not None and step >= max_batches:
            break
        images = batch["images"].to(device, non_blocking=True)
        task_ids = batch["task_ids"].to(device)
        labels = [target.to(device) for target in batch["labels"]]

        if gate is not None:
            with torch.no_grad():
                lung_masks = gate(images)
                images = apply_dilated_soft_mask_tensor(images, lung_masks)

        with amp_autocast():
            outputs = model(images)
            loss, metrics = criterion(outputs, labels, task_ids)
            accum_loss = loss / grad_accum_steps

        scaler.scale(accum_loss).backward()

        if (step + 1) % grad_accum_steps == 0 or (step + 1) == len(dataloader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            if ema is not None:
                ema.update(model)

        batch_size = images.size(0)
        running_total += float(loss.detach().item()) * batch_size
        count += batch_size
        for key in running_heads:
            running_heads[key] += metrics.get(key, 0.0) * batch_size
        if step == 0:
            print(f"  [{epoch_name}] step {step + 1}: loss={float(loss.detach().item()):.4f} "
                  f"(A={metrics.get('loss_a', 0.0):.4f}, B={metrics.get('loss_b', 0.0):.4f}, C={metrics.get('loss_c', 0.0):.4f})")
    averaged = {key: value / max(1, count) for key, value in running_heads.items()}
    return running_total / max(1, count), averaged


def cpu_state_dict(model):
    return {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}


def save_payload(state, epoch, stage, selection_metric, val_metrics, thresholds=None, test_metrics=None):
    payload = {
        "model_state_dict": state,
        "arch": "DenseNet121-AG-MTCNet-3Head",
        "num_classes_c": NUM_CLASSES_C,
        "img_size": IMG_SIZE,
        "config": {
            "seed": SEED,
            "batch_size": BATCH_SIZE,
            "grad_accum_steps": GRAD_ACCUM_STEPS,
            "samples_per_epoch": SAMPLES_PER_EPOCH,
            "batches_per_epoch": BATCHES_PER_EPOCH,
            "task_ratios": TASK_RATIOS,
            "profile": TRAINING_PROFILE,
            "asl": {"gamma_pos": 0.0, "gamma_neg": 4.0, "clip": 0.05},
            "head_c_ce_weights": np.round(ce_weights, 6).tolist(),
            "head_b_pos_weight": tb_pos_weight,
            "epochs_stage1": EPOCHS_STAGE1,
            "epochs_stage2": EPOCHS_STAGE2,
            "ema": USE_EMA,
            "ema_decay": EMA_DECAY,
            "normalization": {"mean": IMAGENET_MEAN.tolist(), "std": IMAGENET_STD.tolist()},
        },
        "metrics": {"val": val_metrics, "test": test_metrics},
        "thresholds": thresholds or {},
        "gate_source": gate_source,
        "data_modes": data_modes,
        "history": history,
        "selection": {"criterion": selection_metric, "epoch": epoch, "stage": stage},
    }
    ckpt_path = ckpt_dir / "ag_mtcnet_best.pth"
    torch.save(payload, ckpt_path)
    if IS_KAGGLE:
        shutil.copy2(ckpt_path, WORK / "ag_mtcnet_best.pth")
    return ckpt_path


print("=== STAGE 1: Encoder frozen, classification heads warmup ===")
for parameter in model.backbone.parameters():
    parameter.requires_grad = False
optimizer_s1 = torch.optim.AdamW(
    [*model.head_a.parameters(), *model.head_b.parameters(), *model.head_c.parameters()],
    lr=1e-4, weight_decay=1e-4,
)

for epoch_s1 in range(1, EPOCHS_STAGE1 + 1):
    sampler.set_epoch(epoch_s1 - 1)
    started = time.perf_counter()
    loss_s1, head_s1 = train_epoch(model, gate_model, train_loader, optimizer_s1, scaler, f"S1-E{epoch_s1}", ema=ema)
    elapsed_s1 = time.perf_counter() - started
    val_s1 = evaluate_heads(val_loaders, MAX_VAL_BATCHES)
    history.append({
        "stage": "S1", "epoch": epoch_s1, "train_loss": loss_s1, **head_s1,
        "val": val_s1, "seconds": elapsed_s1,
    })
    print(f"[S1] Epoch {epoch_s1}/{EPOCHS_STAGE1}: loss={loss_s1:.4f} | val_score={val_s1['score']:.4f} "
          f"| aucA={val_s1['auc_a']:.4f} aucB={val_s1['auc_b']:.4f} aucC={val_s1['auc_c']:.4f} | {elapsed_s1:.1f}s")

stage1_state = cpu_state_dict(model)
stage1_score = float(val_s1["score"])
stage1_metrics = val_s1

print("=== STAGE 2: Unfrozen network with layer-wise differential learning rates ===")
for parameter in model.parameters():
    parameter.requires_grad = True
optimizer_s2 = torch.optim.AdamW(
    build_param_groups(model, backbone_lr=1e-5, head_lr=1e-4, weight_decay=1e-4, decay=0.85)
)
scheduler_s2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_s2, T_max=max(1, EPOCHS_STAGE2), eta_min=1e-7
)

best_score = float("-inf")
best_epoch = 0
best_state = None
best_metrics = None
best_source = "raw"
patience_counter = 0

for epoch in range(1, EPOCHS_STAGE2 + 1):
    sampler.set_epoch(EPOCHS_STAGE1 + epoch)
    started = time.perf_counter()
    loss_s2, head_s2 = train_epoch(
        model, gate_model, train_loader, optimizer_s2, scaler, f"S2-E{epoch}", ema=ema
    )
    scheduler_s2.step()
    elapsed = time.perf_counter() - started

    candidates = []
    raw_metrics = evaluate_heads(val_loaders, MAX_VAL_BATCHES)
    candidates.append(("raw", raw_metrics, cpu_state_dict(model)))
    if ema is not None:
        ema.store(model)
        ema.copy_to(model)
        ema_metrics = evaluate_heads(val_loaders, MAX_VAL_BATCHES)
        candidates.append(("ema", ema_metrics, cpu_state_dict(model)))
        ema.restore(model)

    source, metrics_epoch, state_epoch = max(candidates, key=lambda item: item[1]["score"])
    score = float(metrics_epoch["score"])
    improved = score > best_score + MIN_DELTA

    history.append({
        "stage": "S2", "epoch": epoch, "train_loss": loss_s2, **head_s2,
        "val": metrics_epoch, "score": score, "weights": source, "seconds": elapsed,
    })

    if improved:
        best_score = score
        best_epoch = epoch
        best_state = state_epoch
        best_metrics = metrics_epoch
        best_source = source
        patience_counter = 0
        save_payload(best_state, epoch, "S2", "val_combined_auc", best_metrics)
        print(f"  [BEST] Epoch {epoch} ({source}): score={score:.4f} | aucA={metrics_epoch['auc_a']:.4f} "
              f"aucB={metrics_epoch['auc_b']:.4f} aucC={metrics_epoch['auc_c']:.4f} | {elapsed:.1f}s")
    else:
        patience_counter += 1
        print(f"  [--] Epoch {epoch} ({source}): score={score:.4f} best={best_score:.4f} "
              f"patience={patience_counter}/{PATIENCE} | {elapsed:.1f}s")
        if patience_counter >= PATIENCE:
            print(f"[*] Early stopping triggered at epoch {epoch} (patience={PATIENCE}).")
            break

if best_state is None or stage1_score > best_score + MIN_DELTA:
    print("[*] Stage 2 did not exceed the Stage 1 validation score; exporting the Stage 1 state.")
    best_state = stage1_state
    best_score = stage1_score
    best_metrics = stage1_metrics
    best_source = "stage1"
    best_epoch = 0

model.load_state_dict(best_state)
print(f"[OK] Selected checkpoint: source={best_source} epoch={best_epoch} val_score={best_score:.4f}")
save_payload(best_state, best_epoch, best_source, "val_combined_auc", best_metrics)

In [ ]:
# ---------------------------------------------------------------------------
# 5. Final validation/test evaluation, threshold calibration and export
# ---------------------------------------------------------------------------
print("=== FINAL VALIDATION (full validation cap) ===")
model.load_state_dict(best_state)
model.eval()
val_full, val_preds = evaluate_heads(val_loaders, max_batches=None, return_predictions=True)
print(f"  val_score: {val_full['score']:.4f}")
print(f"  auc_head_a_macro: {val_full['auc_a']:.4f}")
print(f"  auc_head_b: {val_full['auc_b']:.4f}")
print(f"  auc_head_c_ovr: {val_full['auc_c']:.4f} | accuracy_head_c: {val_full['acc_c']:.4f}")

y_a, p_a = val_preds["a"]
y_b, p_b = val_preds["b"]
thresholds_a = tune_youden(y_a, p_a, NIH_PATHOLOGY_LIST) if np.asarray(y_a).size else {}
thresholds_b = tune_youden(y_b.reshape(-1, 1), p_b.reshape(-1, 1), ["tb"]) if np.asarray(y_b).size else {}
print(f"[*] Youden thresholds (Head A, {len(thresholds_a)}/14 classes with both labels): {thresholds_a}")
print(f"[*] Youden threshold (Head B): {thresholds_b}")

test_metrics = None
if any(loader is not None for loader in test_loaders.values()):
    print("=== HELD-OUT TEST EVALUATION (never used for selection) ===")
    test_metrics = evaluate_heads(test_loaders, max_batches=None)
    print(f"  auc_head_a_macro: {test_metrics['auc_a']:.4f}")
    print(f"  auc_head_b: {test_metrics['auc_b']:.4f}")
    print(f"  auc_head_c_ovr: {test_metrics['auc_c']:.4f} | accuracy_head_c: {test_metrics['acc_c']:.4f}")
else:
    print("[!] No held-out test pools available; test metrics are omitted.")

threshold_payload = {
    "head_a_nih_14": thresholds_a,
    "head_b_tb": thresholds_b,
    "head_c_covid": {
        "strategy": "argmax",
        "classes": ["Normal", "Non-COVID Pneumonia", "COVID-19"],
    },
    "img_size": IMG_SIZE,
    "selection": {"criterion": "val_combined_auc", "score": best_score, "source": best_source, "epoch": best_epoch},
    "normalization": {"mean": IMAGENET_MEAN.tolist(), "std": IMAGENET_STD.tolist()},
}
threshold_path = ckpt_dir / "ag_mtcnet_thresholds.json"
with open(threshold_path, "w", encoding="utf-8") as handle:
    json.dump(threshold_payload, handle, indent=2)
if IS_KAGGLE:
    shutil.copy2(threshold_path, WORK / "ag_mtcnet_thresholds.json")

ckpt_path = save_payload(
    best_state, best_epoch, best_source, "val_combined_auc",
    val_full, thresholds={"head_a": thresholds_a, "head_b": thresholds_b},
    test_metrics=test_metrics,
)
print(f"[OK] Final checkpoint: {ckpt_path} ({ckpt_path.stat().st_size / (1024 ** 2):.1f} MB)")
print(f"[OK] Thresholds: {threshold_path}")

try:
    import matplotlib.pyplot as plt

    train_losses = [entry["train_loss"] for entry in history]
    val_scores = [entry["val"]["score"] for entry in history]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(train_losses, marker="o")
    axes[0].set_title("Masked Multi-Task Training Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True, alpha=0.3)
    axes[1].plot(val_scores, marker="o")
    axes[1].set_title("Validation Combined AUC")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Score")
    axes[1].grid(True, alpha=0.3)
    fig.tight_layout()
    curve_path = ckpt_dir / "ag_mtcnet_training_curves.png"
    fig.savefig(curve_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] Training curves: {curve_path}")
except Exception as exc:
    print(f"[!] Curve export skipped: {exc}")

model.eval()
if val_loaders["head_a"] is not None:
    with torch.no_grad():
        sample_batch = next(iter(val_loaders["head_a"]))
        sample_images = sample_batch["images"][:1].to(device)
        if gate_model is not None:
            sample_images = apply_dilated_soft_mask_tensor(sample_images, gate_model(sample_images))
        predictions = model.predict(sample_images)
        assert predictions["head_a"].shape == (1, 14)
        assert predictions["head_b"].shape == (1, 1)
        assert predictions["head_c"].shape == (1, NUM_CLASSES_C)
        for tensor in predictions.values():
            assert torch.isfinite(tensor).all()
    print("[OK] Inference contract validated across all 3 heads (finite probabilities, correct shapes).")

print("\n=== STAGE 2 HAND-OFF SUMMARY ===")
print(f"Balanced epoch : {actual[0]} NIH : {actual[1]} COVID : {actual[2]} TB ({sum(actual)} samples)")
print(f"Data modes     : {' | '.join(data_modes)}")
print(f"Gate           : {gate_source if gate_source else 'identity (Stage 1 checkpoint not found)'}")
print(f"Selection      : {best_source} epoch {best_epoch} (val combined AUC = {best_score:.4f})")
print(f"Validation     : {val_full}")
print(f"Test           : {test_metrics}")
print(f"Checkpoint     : {ckpt_path}")
print(f"Thresholds     : {threshold_path}")

## Stage 2 completion criteria

- **Rule 2/3 (Zero Leakage)**: NIH is partitioned by `Patient_ID`; COVIDx is partitioned and stratified by patient and clinical source. Patient isolation is asserted before training. The legacy Radiography CSV cannot provide patient IDs; the notebook prints an explicit warning when that fallback is used.
- **Rule 4 (Bounding Box)**: no bounding box is read. Supervised heads and WSOL learn from image-level labels only.
- **Rule 5 (Balanced Batch)**: every minibatch follows the configured 60 : 25 : 15 task ratio; the exact per-epoch counts are derived from `SAMPLES_PER_EPOCH` and asserted at runtime.
- **Rule 6 (Guardrail readiness)**: the frozen Dense-UNet gate used during training is recorded in the checkpoint payload (`gate_source`). If the gate is missing, the notebook prints a loud warning and the checkpoint records `identity`.
- **Best-checkpoint contract**: `checkpoints/ag_mtcnet_best.pth` contains the state selected by validation combined AUC, not the last training epoch. Both raw and EMA candidates are evaluated; the winner is exported.
- **Deployment artifacts**: `ag_mtcnet_best.pth` (weights + config + metrics + thresholds), `ag_mtcnet_thresholds.json` (Youden-J thresholds), and `ag_mtcnet_training_curves.png`.

### Loading the exported checkpoint

```python
import torch
from src.models.multi_task_net import MultiTaskChestXrayNet

payload = torch.load("checkpoints/ag_mtcnet_best.pth", map_location="cpu")
model = MultiTaskChestXrayNet(
    backbone_name="densenet121", pretrained=False, img_size=payload["img_size"],
    head_norm="layernorm", num_classes_c=payload["num_classes_c"],
)
model.load_state_dict(payload["model_state_dict"])
model.eval()
thresholds = payload["thresholds"]
metrics = payload["metrics"]
```